In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, when,
    date_format, year, month, quarter, dayofweek, dayofmonth, 
    weekofyear, round, min, max,
    to_date
)

In [0]:
spark = SparkSession.builder.appName("TransformacaoGold").getOrCreate()

spark.sql("CREATE DATABASE IF NOT EXISTS gold")
spark.sql("USE gold")

ps: nessa atividade decidi criar as variaveis logo no inicio do código para facilitar manutencao e rastreabilidade 

In [0]:
db_silver = "silver"
db_gold = "gold"

silver_ft_pedido_total = f"{db_silver}.ft_pedido_total"
silver_ft_consumidores = f"{db_silver}.ft_consumidores"
silver_ft_pedidos = f"{db_silver}.ft_pedidos"
silver_ft_itens_pedidos = f"{db_silver}.ft_itens_pedidos"
silver_ft_produtos = f"{db_silver}.ft_produtos"
silver_dm_cotacao_dolar = f"{db_silver}.dm_cotacao_dolar"
silver_ft_avaliacoes_pedidos = f"{db_silver}.ft_avaliacoes_pedidos"

gold_ft_vendas_consumidor_local = f"{db_gold}.ft_vendas_consumidor_local"
gold_ft_atrasos_pedidos_local_vendedor = f"{db_gold}.ft_atrasos_pedidos_local_vendedor"
gold_dm_tempo = f"{db_gold}.dm_tempo"
gold_dm_produtos = f"{db_gold}.dm_produtos"
gold_ft_vendas_geral = f"{db_gold}.ft_vendas_geral"

gold_view_total_compras_por_consumidor = f"{db_gold}.view_total_compras_por_consumidor"
gold_view_tempo_medio_entrega_localidade = f"{db_gold}.view_tempo_medio_entrega_localidade"
gold_view_vendedor_pontualidade = f"{db_gold}.view_vendedor_pontualidade"
gold_view_vendas_por_periodo = f"{db_gold}.view_vendas_por_periodo"
gold_view_top_produto = f"{db_gold}.view_top_produto"
gold_view_vendas_produtos_esteticos = f"{db_gold}.view_vendas_produtos_esteticos"

## Projeto 1: Logística (Vendas por Localidade)

**Objetivo:** Identificar cidades e estados com maior concentração de vendas para otimização de rotas e centros de distribuição.

### 1.1 Tabela Fato: `ft_vendas_consumidor_local`
Cada linha representa um pedido com informações de localização do consumidor.

In [0]:
df_pedido_total = spark.read.table(silver_ft_pedido_total)
df_consumidores = spark.read.table(silver_ft_consumidores)

df_vendas_consumidor_local = df_pedido_total.join(
    df_consumidores,
    "id_consumidor",
    "inner"
).select(
    col("id_pedido"),
    col("id_consumidor"),
    col("valor_total_pago_brl").alias("valor_total_pedido_brl").cast("decimal(12,2)"),
    col("cidade"),
    col("estado"),
    col("data_pedido")
)

df_vendas_consumidor_local.write.mode("overwrite").saveAsTable(gold_ft_vendas_consumidor_local)

In [0]:
query = f"""
CREATE OR REPLACE VIEW {gold_view_total_compras_por_consumidor} AS
SELECT
  cidade,
  estado,
  COUNT(id_pedido) AS quantidade_vendas,
  SUM(valor_total_pedido_brl) AS valor_total_localidade
FROM
  {gold_ft_vendas_consumidor_local}
GROUP BY
  cidade,
  estado
"""

spark.sql(query)

respondendo perguntas da area de negocio total de vendas por estado


In [0]:
query = f"""
SELECT
  estado,
  SUM(valor_total_localidade) AS total_vendas_estado
FROM
  {gold_view_total_compras_por_consumidor}
GROUP BY
  estado
ORDER BY
  total_vendas_estado DESC
"""

spark.sql(query).show()

## Projeto 2: Logística (Análise de Atrasos de Entregas)

**Objetivo:** Identificar regiões e vendedores com maiores índices de atraso para detectar gargalos na cadeia logística.

### 2.1 Tabela Fato: `ft_atrasos_pedidos_local_vendedor`
Cada linha representa um pedido com informações logísticas (vendedor, localização, prazos).

In [0]:
df_pedidos = spark.read.table(silver_ft_pedidos)
df_consumidores = spark.read.table(silver_ft_consumidores)
df_itens_pedidos = spark.read.table(silver_ft_itens_pedidos)

df_atrasos_base = df_pedidos.join(
    df_consumidores,
    "id_consumidor",
    "inner"
).join(
    df_itens_pedidos,
    "id_pedido",
    "inner"
).select(
    df_pedidos.id_pedido,
    df_itens_pedidos.id_vendedor,
    df_consumidores.id_consumidor,
    df_pedidos.entrega_no_prazo,
    df_pedidos.tempo_entrega_dias,
    df_pedidos.tempo_entrega_estimado_dias,
    df_consumidores.cidade,
    df_consumidores.estado
).dropDuplicates(["id_pedido", "id_vendedor"])

df_atrasos_base.write.mode("overwrite").saveAsTable(gold_ft_atrasos_pedidos_local_vendedor)

### 2.2 Views Analíticas

#### 2.2.1 View: `view_tempo_medio_entrega_localidade`
Calcula tempo médio de entrega por cidade/estado e identifica localidades com entrega acima do estimado.

In [0]:
query = f"""
CREATE OR REPLACE VIEW {gold_view_tempo_medio_entrega_localidade} AS
SELECT
  cidade,
  estado,
  AVG(tempo_entrega_dias) AS tempo_medio_entrega,
  AVG(tempo_entrega_estimado_dias) AS tempo_medio_estimado,
  CASE
    WHEN AVG(tempo_entrega_dias) > AVG(tempo_entrega_estimado_dias) THEN 'SIM'
    ELSE 'NÃO'
  END AS entrega_maior_que_estimado
FROM
  {gold_ft_atrasos_pedidos_local_vendedor}
GROUP BY
  cidade,
  estado
"""

spark.sql(query)

#### 2.2.2 View: `view_vendedor_pontualidade`
Métricas de pontualidade por vendedor (total de pedidos, atrasos e percentual).

In [0]:
query = f"""
CREATE OR REPLACE VIEW {gold_view_vendedor_pontualidade} AS
SELECT
  id_vendedor,
  COUNT(id_pedido) AS total_pedidos,
  SUM(CASE WHEN entrega_no_prazo = 'Não' THEN 1 ELSE 0 END) AS total_atrasados,
  (SUM(CASE WHEN entrega_no_prazo = 'Não' THEN 1 ELSE 0 END) / COUNT(id_pedido)) * 100 AS percentual_atraso
FROM
  {gold_ft_atrasos_pedidos_local_vendedor}
GROUP BY
  id_vendedor
"""

spark.sql(query)

## Projeto 3: Comercial (Análises de Vendas por Período)

**Objetivo:** Acompanhar evolução de vendas com análises temporais e de desempenho para suporte a decisões comerciais.

### 3.1 Dimensão Temporal: `dm_tempo`
Criada usando `explode()` e `sequence()` para gerar todas as datas entre o período mínimo e máximo dos pedidos.

In [0]:
min_max_date = spark.read.table(silver_ft_pedidos).select(
    min(to_date(col("pedido_compra_timestamp"))).alias("min_date"),
    max(to_date(col("pedido_compra_timestamp"))).alias("max_date")
).first()

min_date = min_max_date["min_date"]
max_date = min_max_date["max_date"]

df_dates = spark.sql(f"SELECT explode(sequence(TO_DATE('{min_date}'), TO_DATE('{max_date}'))) AS sk_tempo")

df_dim_tempo = df_dates.select(
    col("sk_tempo"),
    year("sk_tempo").alias("ano"),
    quarter("sk_tempo").alias("trimestre"),
    month("sk_tempo").alias("mes"),
    weekofyear("sk_tempo").alias("semana_do_ano"),
    dayofmonth("sk_tempo").alias("dia"),
    dayofweek("sk_tempo").alias("dia_da_semana_num"),
    date_format("sk_tempo", "EEEE").alias("dia_da_semana_nome"),
    date_format("sk_tempo", "MMMM").alias("mes_nome"),
    when(dayofweek("sk_tempo").isin([1, 7]), "Sim").otherwise("Não").alias("eh_fim_de_semana")
)

df_dim_tempo.write.mode("overwrite").saveAsTable(gold_dm_tempo)

### 3.1.1 Dimensão de Produtos: `dm_produtos`
Senti a necessidade de criar a dimensao de produtos para evitar consumir na CTE direto da camada silver , agora com essa dimensao consumo diretamente da gold

Tabela dimensional com informações de produtos para uso na camada Gold.

In [0]:
df_produtos = spark.read.table(silver_ft_produtos)

df_dm_produtos = df_produtos.select(
    col("id_produto"),
    col("categoria_produto"),
    col("peso_produto_gramas")
).dropDuplicates(["id_produto"])

df_dm_produtos.write.mode("overwrite").saveAsTable(gold_dm_produtos)

### 3.2 Tabela Fato: `ft_vendas_geral`
Integra informações de múltiplas áreas (vendas, produtos, clientes, cotação dólar) com valores em BRL e USD.

In [0]:
df_itens = spark.read.table(silver_ft_itens_pedidos)
df_pedidos = spark.read.table(silver_ft_pedidos)
df_consumidores = spark.read.table(silver_ft_consumidores)
df_cotacao = spark.read.table(silver_dm_cotacao_dolar)
df_avaliacoes = spark.read.table(silver_ft_avaliacoes_pedidos)

df_avaliacoes_agg = df_avaliacoes.groupBy("id_pedido").agg(
    avg("avaliacao").alias("avaliacao_media")
)

df_base_vendas = df_itens.join(
    df_pedidos,
    "id_pedido",
    "inner"
).join(
    df_consumidores,
    "id_consumidor",
    "inner"
).join(
    df_cotacao,
    to_date(df_pedidos["pedido_compra_timestamp"]) == df_cotacao["data"],
    "left"
).join(
    df_avaliacoes_agg,
    "id_pedido",
    "left"
)

df_ft_vendas_geral = df_base_vendas.select(
    df_itens.id_pedido,
    df_itens.id_item,
    df_consumidores.id_consumidor.alias("fk_cliente"),
    df_itens.id_produto.alias("fk_produto"),
    df_itens.id_vendedor.alias("fk_vendedor"),
    to_date(df_pedidos.pedido_compra_timestamp).alias("fk_tempo"),
    df_pedidos.status.alias("status_pedido"),
    df_pedidos.tempo_entrega_dias,
    df_pedidos.entrega_no_prazo,
    df_itens.preco_BRL.alias("valor_produto_brl"),
    df_itens.preco_frete.alias("valor_frete_brl"),
    (df_itens.preco_BRL + df_itens.preco_frete).alias("valor_total_item_brl").cast("decimal(12,2)"),
    round(df_itens.preco_BRL / df_cotacao.cotacao_dolar, 2).alias("valor_produto_usd").cast("decimal(12,2)"),
    round(df_itens.preco_frete / df_cotacao.cotacao_dolar, 2).alias("valor_frete_usd").cast("decimal(12,2)"),
    round((df_itens.preco_BRL + df_itens.preco_frete) / df_cotacao.cotacao_dolar, 2).alias("valor_total_item_usd").cast("decimal(12,2)"),
    df_cotacao.cotacao_dolar.cast("decimal(8,4)"),
    df_avaliacoes_agg.avaliacao_media.alias("avaliacao_pedido").cast("decimal(3,2)")
)

df_ft_vendas_geral.write.mode("overwrite").saveAsTable(gold_ft_vendas_geral)

### 3.3 View: `view_vendas_por_periodo`
Análise temporal consolidada com métricas de vendas agregadas por ano, trimestre, mês e dia.

In [0]:
query = f"""
CREATE OR REPLACE VIEW {gold_view_vendas_por_periodo} AS
SELECT
  t.ano,
  t.trimestre,
  t.mes,
  t.mes_nome,
  t.dia,
  t.dia_da_semana_num,
  t.dia_da_semana_nome,
  COUNT(DISTINCT v.id_pedido) AS total_pedidos,
  COUNT(v.id_item) AS total_itens,
  SUM(v.valor_total_item_brl) AS receita_total_brl,
  SUM(v.valor_total_item_usd) AS receita_total_usd,
  AVG(v.valor_total_item_brl) AS ticket_medio_brl,
  AVG(v.avaliacao_pedido) AS avaliacao_media
FROM
  {gold_ft_vendas_geral} v
JOIN
  {gold_dm_tempo} t ON v.fk_tempo = t.sk_tempo
GROUP BY
  t.ano,
  t.trimestre,
  t.mes,
  t.mes_nome,
  t.dia,
  t.dia_da_semana_num,
  t.dia_da_semana_nome
"""

spark.sql(query)

In [0]:
%sql
SELECT * FROM gold.view_vendas_por_periodo

### 3.3.1 Queries Analíticas

Duas análises para responder perguntas estratégicas do time comercial.

In [0]:
query_1 = f"""
SELECT
  dia_da_semana_nome,
  SUM(receita_total_brl) AS receita_total
FROM
  {gold_view_vendas_por_periodo}
GROUP BY
  dia_da_semana_nome
ORDER BY
  receita_total DESC
LIMIT 1
"""

print("QUERY 1: Dia da semana com maior receita total em reais")
spark.sql(query_1).show()

query_2 = f"""
WITH ultimo_ano AS (
  SELECT MAX(ano) AS max_ano FROM {gold_dm_tempo}
),
vendas_ultimo_ano AS (
  SELECT
    mes,
    mes_nome,
    SUM(receita_total_brl) AS receita_total_mes,
    SUM(total_pedidos) AS total_pedidos_mes
  FROM
    {gold_view_vendas_por_periodo}
  WHERE
    ano = (SELECT max_ano FROM ultimo_ano)
  GROUP BY
    mes, mes_nome
)
SELECT
  mes_nome,
  ROUND(receita_total_mes / total_pedidos_mes, 2) AS ticket_medio_mensal
FROM
  vendas_ultimo_ano
ORDER BY
  ticket_medio_mensal DESC
LIMIT 1
"""

print("QUERY 2: Mês com maior ticket médio no último ano")
spark.sql(query_2).show()

### 3.4 View: `view_top_produto`
Métricas de performance por produto (receita, quantidade, avaliação).

In [0]:
query = f"""
CREATE OR REPLACE VIEW {gold_view_top_produto} AS
SELECT
  v.fk_produto AS id_produto,
  p.categoria_produto,
  COUNT(v.id_item) AS quantidade_vendida,
  COUNT(DISTINCT v.id_pedido) AS total_pedidos,
  SUM(v.valor_total_item_brl) AS receita_brl,
  SUM(v.valor_total_item_usd) AS receita_usd,
  AVG(v.valor_produto_brl) AS preco_medio_brl,
  AVG(v.avaliacao_pedido) AS avaliacao_media,
  AVG(p.peso_produto_gramas) AS peso_medio_gramas
FROM
  {gold_ft_vendas_geral} v
JOIN
  {gold_dm_produtos} p ON v.fk_produto = p.id_produto
GROUP BY
  v.fk_produto,
  p.categoria_produto
"""

spark.sql(query)

### 3.5 View: `view_vendas_produtos_esteticos`
Análise temporal de produtos fashion com uso obrigatório de CTE.

In [0]:
query = f"""
CREATE OR REPLACE VIEW {gold_view_vendas_produtos_esteticos} AS
WITH VendasFashion AS (
  SELECT
    t.ano,
    t.mes,
    p.categoria_produto,
    v.id_pedido,
    v.id_item,
    v.valor_total_item_brl,
    v.valor_total_item_usd,
    v.avaliacao_pedido
  FROM
    {gold_ft_vendas_geral} v
  JOIN
    {gold_dm_produtos} p ON v.fk_produto = p.id_produto
  JOIN
    {gold_dm_tempo} t ON v.fk_tempo = t.sk_tempo
  WHERE
    p.categoria_produto LIKE 'fashion%'
)
SELECT
  ano,
  mes,
  categoria_produto,
  COUNT(DISTINCT id_pedido) AS total_pedidos,
  COUNT(id_item) AS total_itens_vendidos,
  SUM(valor_total_item_brl) AS receita_total_brl,
  SUM(valor_total_item_usd) AS receita_total_usd,
  AVG(valor_total_item_brl) AS ticket_medio_brl,
  AVG(valor_total_item_usd) AS ticket_medio_usd,
  AVG(avaliacao_pedido) AS avaliacao_media
FROM
  VendasFashion
GROUP BY
  ano,
  mes,
  categoria_produto
"""

spark.sql(query)